# Dataset Verification and Manifest Generation

Scans the raw FFE directory tree, checks image integrity, and writes a labeled manifest CSV consumed by every downstream notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'

import os
os.makedirs('/content/data', exist_ok=True)
!unzip -q "$DATASET_ZIP" -d /content/data
!find "$DATASET_ROOT" -maxdepth 1 -type d | wc -l

In [ ]:
import sys
sys.path.append('/content/repo/04_Src')

from manifest_utils import build_manifest, summarize_class_distribution

df = build_manifest(DATASET_ROOT)
print(f'total images: {len(df)}')
df.head()

In [ ]:
summarize_class_distribution(df)

In [ ]:
EXPECTED_TOTAL = 4390
assert len(df) == EXPECTED_TOTAL, f'expected {EXPECTED_TOTAL} images, found {len(df)}'
print('image count matches expected total')

## Integrity check: corrupt and duplicate files

In [ ]:
import hashlib
from pathlib import Path
from PIL import Image, UnidentifiedImageError

corrupt = []
hashes = {}
duplicates = []

for rel_path in df['filepath']:
    path = Path(DATASET_ROOT) / rel_path
    try:
        with Image.open(path) as im:
            im.verify()
    except (UnidentifiedImageError, OSError):
        corrupt.append(rel_path)
        continue
    with open(path, 'rb') as f:
        h = hashlib.md5(f.read()).hexdigest()
    if h in hashes:
        duplicates.append((rel_path, hashes[h]))
    else:
        hashes[h] = rel_path

print(f'corrupt files: {len(corrupt)}')
print(f'duplicate files: {len(duplicates)}')

In [ ]:
OUT_PATH = '/content/repo/02_Manifests/manifest_full.csv'
df.to_csv(OUT_PATH, index=False)
print('saved to', OUT_PATH)